In [ ]:
# 1. Injeta os caminhos do CUDA nas variáveis de ambiente do sistema
import os
os.environ['PATH'] += ':/usr/local/cuda/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda/lib64'

# 2. Força o Colab a voltar para a pasta raiz segura do sistema
%cd /content

# 3. Garante que qualquer versão antiga ou duplicada seja apagada antes do clone
!rm -rf Algebra-linear-multiplicacao-de-matrizes-densas-GEMM

# 4. Clona o repositório original de forma limpa
!git clone https://github.com/guidias-sketch/Algebra-linear-multiplicacao-de-matrizes-densas-GEMM.git

# 5. Entra na pasta do projeto
%cd Algebra-linear-multiplicacao-de-matrizes-densas-GEMM

# 6. Compila os códigos C++/CUDA usando o Makefile
!make

In [ ]:
import numpy as np
import ctypes
import time
import matplotlib.pyplot as plt

# Carrega a biblioteca compartilhada compilada na célula anterior
lib = ctypes.CDLL('/content/Algebra-linear-multiplicacao-de-matrizes-densas-GEMM/libgemm.so')

# Ajustados os tipos de argumentos para receber as referências de tempo precisas do C++
lib.run_benchmarks.argtypes = [
    ctypes.c_int,
    ctypes.POINTER(ctypes.c_float), # h_A
    ctypes.POINTER(ctypes.c_float), # h_B
    ctypes.POINTER(ctypes.c_float), # h_C_cpu
    ctypes.POINTER(ctypes.c_float), # h_C_naive
    ctypes.POINTER(ctypes.c_float), # h_C_tiled
    ctypes.POINTER(ctypes.c_float), # t_cpu (Saída)
    ctypes.POINTER(ctypes.c_float), # t_naive (Saída)
    ctypes.POINTER(ctypes.c_float)  # t_tiled (Saída)
]

def benchmark_gemm(N=512): 
    print(f"Iniciando benchmarks reais para matriz {N}x{N}...\n")
    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)
    
    C_cpu = np.zeros((N, N), dtype=np.float32)
    C_naive = np.zeros((N, N), dtype=np.float32)
    C_tiled = np.zeros((N, N), dtype=np.float32)

    # Variáveis de destino para receber os tempos calculados pelo C++/CUDA
    t_cpu = ctypes.c_float(0.0)
    t_naive = ctypes.c_float(0.0)
    t_tiled = ctypes.c_float(0.0)

    # 1. Benchmark NumPy de Referência (cuBLAS/MKL)
    start = time.time()
    C_numpy = np.matmul(A, B)
    t_numpy = time.time() - start
    print(f"⏱️ Tempo NumPy (Referência): {t_numpy:.4f}s")

    # Preparando Ponteiros para o ctypes
    ptr_A = A.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_B = B.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_cpu = C_cpu.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_naive = C_naive.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_tiled = C_tiled.ctypes.data_as(ctypes.POINTER(ctypes.c_float))

    # Execução do Backend com cronometragem nativa de hardware
    lib.run_benchmarks(
        N, ptr_A, ptr_B, ptr_C_cpu, ptr_C_naive, ptr_C_tiled,
        ctypes.byref(t_cpu), ctypes.byref(t_naive), ctypes.byref(t_tiled)
    )

    # Extrai valores numéricos de ponto flutuante das referências
    v_cpu = t_cpu.value
    v_naive = t_naive.value
    v_tiled = t_tiled.value

    # Validação matemática de integridade estrita
    np.testing.assert_allclose(C_numpy, C_cpu, atol=1e-2)
    np.testing.assert_allclose(C_numpy, C_naive, atol=1e-2)
    np.testing.assert_allclose(C_numpy, C_tiled, atol=1e-2)
    print("✅ Sucesso! Todas as matrizes calculadas batem perfeitamente com o NumPy.")

    # Exibição dos resultados textuais obtidos diretamente do hardware
    print("\n--- ANÁLISE DE TEMPOS REAIS CORRIGIDA ---")
    print(f"a) CPU Sequencial: {v_cpu:.6f}s")
    print(f"b) CUDA Naive (Memória Global): {v_naive:.6f}s")
    print(f"c) CUDA Tiled (Shared Memory): {v_tiled:.6f}s")
    
    if v_tiled > 0:
        print(f"🚀 Ganho da Hierarquia de Memória (Tiled vs Naive): {v_naive/v_tiled:.2f}x de Speedup!")

    # GERAÇÃO DO GRÁFICO DO RELATÓRIO
    metodos = ['CPU Seq', 'NumPy (Ref)', 'CUDA Naive', 'CUDA Tiled']
    tempos = [v_cpu, t_numpy, v_naive, v_tiled]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(metodos, tempos, color=['#e74c3c', '#2ecc71', '#3498db', '#9b59b6'])
    plt.ylabel('Tempo de Execução em Segundos (Escala Logarítmica)')
    plt.title(f'Análise de Desempenho GEMM - Matrizes Reais {N}x{N}')
    plt.yscale('log') # Escala logarítmica devido à disparidade de performance
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval, f'{yval:.6f}s', ha='center', va='bottom', fontweight='bold')
        
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.show()

# Executa o teste
benchmark_gemm(512)